In [ ]:
%%capture
!pip install -q optuna-integration[botorch] optuna botorch gpytorch numba numpy scipy matplotlib

In [3]:
import numpy as np
import os
import numba
from numba import njit
import torch
from optuna.storages import RDBStorage
import optuna
from optuna_integration.botorch import BoTorchSampler
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.acquisition.analytic import AnalyticAcquisitionFunction
from botorch.optim import optimize_acqf
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from scipy.stats import norm
from scipy.optimize import differential_evolution, NonlinearConstraint
import scipy.io as sio
import matplotlib.pyplot as plt
import warnings

# Suppress warnings to keep the Colab output clean
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

In [4]:
# =============================================================================
# 1. NUMBA GRAPH UTILITIES & INCREMENTAL METRICS
# =============================================================================

@njit
def popcount64(x):
    """Extremely fast pure-Python popcount for 64-bit integers."""
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return int((x * np.uint64(0x0101010101010101)) >> np.uint64(56))

@njit
def init_random_graph(rho, N=100):
    """Initializes a random graph and returns adjacency bitsets."""
    words_per_row = (N + 63) // 64
    adj = np.zeros((N, words_per_row), dtype=np.uint64)
    for i in range(N):
        for j in range(i + 1, N):
            if np.random.rand() < rho:
                adj[i, j // 64] |= (np.uint64(1) << np.uint64(j % 64))
                adj[j, i // 64] |= (np.uint64(1) << np.uint64(i % 64))
    return adj

@njit
def compute_metrics(adj, part, N=100):
    """Computes all exact metrics from scratch (used only at initialization)."""
    words = (N + 63) // 64
    E = 0
    degrees = np.zeros(N, dtype=np.int32)
    sum_k = 0
    sum_k2 = 0
    for i in range(N):
        k = 0
        for w in range(words):
            k += popcount64(adj[i, w])
        degrees[i] = k
        E += k
        sum_k += k
        sum_k2 += k * k
    E //= 2

    T = 0
    for i in range(N):
        for j in range(i + 1, N):
            if (adj[i, j // 64] & (np.uint64(1) << np.uint64(j % 64))) > 0:
                c_val = 0
                for w in range(words):
                    c_val += popcount64(adj[i, w] & adj[j, w])
                T += c_val
    T //= 3

    P = sum_k2 - 2 * E

    cross_edges = 0
    for i in range(N):
        for j in range(i + 1, N):
            if part[i] != part[j]:
                if (adj[i, j // 64] & (np.uint64(1) << np.uint64(j % 64))) > 0:
                    cross_edges += 1

    return E, degrees, sum_k, sum_k2, T, P, cross_edges

@njit
def greedy_bisection(adj, current_part, N=100):
    """
    Kernighan-Lin style bisection heuristic.
    Because we only swap opposite-partition nodes, the 50/50 bisection ratio is perfectly maintained.
    """
    part = current_part.copy()
    words = (N + 63) // 64

    while True:
        best_gain = -999999
        best_u = -1
        best_v = -1

        # Calculate D[i] = external degree - internal degree
        D = np.zeros(N, dtype=np.int32)
        for i in range(N):
            ext = 0
            int_ = 0
            for j in range(N):
                if i == j: continue
                is_edge = (adj[i, j // 64] & (np.uint64(1) << np.uint64(j % 64))) > 0
                if is_edge:
                    if part[i] != part[j]: ext += 1
                    else: int_ += 1
            D[i] = ext - int_

        # Find best swap
        for u in range(N):
            if part[u] == 0:
                for v in range(N):
                    if part[v] == 1:
                        is_edge = (adj[u, v // 64] & (np.uint64(1) << np.uint64(v % 64))) > 0
                        gain = D[u] + D[v] - 2 * (1 if is_edge else 0)
                        if gain > best_gain:
                            best_gain = gain
                            best_u = u
                            best_v = v

        if best_gain > 0:
            part[best_u] = 1
            part[best_v] = 0
        else:
            break

    # Count updated cross edges
    cross = 0
    for i in range(N):
        for j in range(i + 1, N):
            if part[i] != part[j]:
                if (adj[i, j // 64] & (np.uint64(1) << np.uint64(j % 64))) > 0:
                    cross += 1
    return part, cross

@njit
def compute_loss(E, sum_k, sum_k2, T, P, cross_edges, target, N=100):
    """Symmetric Euclidean Distance for mapping feasibility."""
    rho_star, eta_star, c_star, mu_star = target

    rho = 2.0 * E / (N * (N - 1.0))

    mean_k = sum_k / float(N)
    if mean_k > 0:
        mean_k2 = sum_k2 / float(N)
        cv2 = (mean_k2 / (mean_k * mean_k)) - 1.0
        eta = cv2 if cv2 < 1.0 else 1.0
    else:
        eta = 0.0

    c = (6.0 * T / P) if P > 0 else 0.0
    mu = (cross_edges / float(E)) if E > 0 else 0.0

    return np.sqrt((rho - rho_star)**2 + (eta - eta_star)**2 + (c - c_star)**2 + (mu - mu_star)**2)


# =============================================================================
# 2. MULTI-START SIMULATED ANNEALING ALGORITHM
# =============================================================================

@njit
def run_sa(target, N=100, restarts=5, steps_per_restart=20000):
    """Explores the parameter space using incremental O(1) state updates."""
    best_loss = 1e9
    words = (N + 63) // 64

    # Cooling schedule: start 0.2, finish at ~1e-4.
    cooling = 0.9996

    for r in range(restarts):
        adj = init_random_graph(target[0], N)

        # Init 50/50 partition randomly
        part = np.zeros(N, dtype=np.int32)
        part[N // 2:] = 1
        for i in range(N - 1, 0, -1):
            j = np.random.randint(0, i + 1)
            part[i], part[j] = part[j], part[i]

        part, cross_edges = greedy_bisection(adj, part, N)
        E, degrees, sum_k, sum_k2, T, P, cross_edges = compute_metrics(adj, part, N)

        loss = compute_loss(E, sum_k, sum_k2, T, P, cross_edges, target, N)
        if loss < best_loss: best_loss = loss

        temp = 0.2

        for step in range(steps_per_restart):
            u = np.random.randint(N)
            v = np.random.randint(N)
            while u == v:
                v = np.random.randint(N)

            mask_v = np.uint64(1) << np.uint64(v % 64)
            mask_u = np.uint64(1) << np.uint64(u % 64)
            is_edge = (adj[u, v // 64] & mask_v) > 0
            delta = -1 if is_edge else 1

            # 1. Triangles & common neighbours
            common = 0
            for w in range(words):
                common += popcount64(adj[u, w] & adj[v, w])
            T_new = T + delta * common

            # 2. Triples & degrees
            old_contrib = degrees[u] * (degrees[u] - 1) + degrees[v] * (degrees[v] - 1)
            new_deg_u = degrees[u] + delta
            new_deg_v = degrees[v] + delta
            new_contrib = new_deg_u * (new_deg_u - 1) + new_deg_v * (new_deg_v - 1)
            P_new = P + new_contrib - old_contrib

            sum_k_new = sum_k + 2 * delta
            sum_k2_new = sum_k2 + (new_deg_u**2 - degrees[u]**2) + (new_deg_v**2 - degrees[v]**2)
            E_new = E + delta

            # 3. Cross edges
            cross_edges_new = cross_edges
            if part[u] != part[v]:
                cross_edges_new += delta

            loss_new = compute_loss(E_new, sum_k_new, sum_k2_new, T_new, P_new, cross_edges_new, target, N)

            # Metropolis-Hastings Acceptance
            if loss_new < loss or np.random.rand() < np.exp((loss - loss_new) / temp):
                loss = loss_new
                E = E_new
                T = T_new
                P = P_new
                sum_k = sum_k_new
                sum_k2 = sum_k2_new
                degrees[u] = new_deg_u
                degrees[v] = new_deg_v
                cross_edges = cross_edges_new

                # XOR flips the bit seamlessly whether adding or removing
                adj[u, v // 64] ^= mask_v
                adj[v, u // 64] ^= mask_u

                if loss < best_loss:
                    best_loss = loss

            # 4. Refresh bisection periodically
            if step % 50 == 0 and step > 0:
                part, cross_edges = greedy_bisection(adj, part, N)
                loss = compute_loss(E, sum_k, sum_k2, T, P, cross_edges, target, N)
                if loss < best_loss: best_loss = loss

            temp *= cooling

    return best_loss


# =============================================================================
# 3. BAYESIAN OPTIMIZATION: BO-TORCH ACQUISITION
# =============================================================================

class LevelSetEstimation(AnalyticAcquisitionFunction):
    """
    Custom Acquisition Function for active level-set boundary learning.
    """
    def __init__(self, model, threshold=0.01):
        super().__init__(model)
        self.threshold = threshold

    def forward(self, X):
        posterior = self.model.posterior(X)
        # AnalyticAcquisitionFunction expects the output to have shape X.shape[:-2]
        # X is (b, 1, d) -> shape is (b,)
        mu = posterior.mean.view(X.shape[:-2])
        sigma = posterior.variance.clamp_min(1e-9).sqrt().view(X.shape[:-2])

        # Maximize variance weighted by proximity to threshold (Straddle rule)
        val = 1.96 * sigma - torch.abs(mu - self.threshold)

        # Guard against NaN extrapolations that crash the multinomial sampler
        return torch.nan_to_num(val, nan=-1e9)

def level_set_candidates_func(train_x, train_obj, train_con, bounds, pending_x):
    """
    Integrates the custom Acquisition Function into Optuna's BoTorchSampler.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_x = train_x.to(device)
    # Ensure correct objective shape (N, 1) and transfer to device
    train_obj = train_obj.view(-1, 1).to(device)
    bounds = bounds.to(device)

    # Train heavily stabilized GP
    model = SingleTaskGP(
        train_x,
        train_obj,
        input_transform=Normalize(d=train_x.shape[-1]),
        outcome_transform=Standardize(m=1)
    ).to(device)

    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)

    acqf = LevelSetEstimation(model, threshold=0.01)

    candidates, _ = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=10,
        raw_samples=512,
    )

    return candidates.cpu()


# =============================================================================
# 4. OPTUNA PIPELINE
# =============================================================================

def objective(trial):
    """Objective function querying the multi-start Simulated Annealer."""
    rho = trial.suggest_float("rho", 0.0, 1.0)
    eta = trial.suggest_float("eta", 0.0, 1.0)
    c   = trial.suggest_float("c",   0.0, 1.0)
    mu  = trial.suggest_float("mu",  0.0, 1.0)

    target = np.array([rho, eta, c, mu], dtype=np.float64)
    loss = run_sa(target, N=100, restarts=5, steps_per_restart=20000)

    return loss

if __name__ == "__main__":
    print("Initializing mapping of 4-D Manifold. Environment setup in progress...")

    sampler = BoTorchSampler(
        candidates_func=level_set_candidates_func,
        n_startup_trials=64
    )

    study = optuna.create_study(direction="minimize", sampler=sampler)

    print("Beginning 500 trials of Bayesian Optimization with Level Set Sampling.")
    print("Note: This execution evaluates millions of graph steps and might take roughly 15-20 minutes.")

    # Optimizing!
    study.optimize(objective, n_trials=500)

[I 2026-06-10 23:32:03,808] A new study created in memory with name: no-name-c1465eff-a85e-404c-bea2-9f422c871551


Initializing mapping of 4-D Manifold. Environment setup in progress...
Beginning 500 trials of Bayesian Optimization with Level Set Sampling.
Note: This execution evaluates millions of graph steps and might take roughly 15-20 minutes.


[I 2026-06-10 23:32:08,054] Trial 0 finished with value: 0.8653824570719827 and parameters: {'rho': 0.2785158633816599, 'eta': 0.828985972936091, 'c': 0.915387433293511, 'mu': 0.5736350692787126}. Best is trial 0 with value: 0.8653824570719827.
[I 2026-06-10 23:32:08,517] Trial 1 finished with value: 0.5997166461548421 and parameters: {'rho': 0.9484437165726882, 'eta': 0.04088075102599997, 'c': 0.11369138077847374, 'mu': 0.27380977296188724}. Best is trial 1 with value: 0.5997166461548421.
[I 2026-06-10 23:32:09,030] Trial 2 finished with value: 0.6707827722107566 and parameters: {'rho': 0.20027910111426794, 'eta': 0.623155534931891, 'c': 0.588191267853797, 'mu': 0.6949286783864272}. Best is trial 1 with value: 0.5997166461548421.
[I 2026-06-10 23:32:09,484] Trial 3 finished with value: 0.4699330660549007 and parameters: {'rho': 0.016037712511302815, 'eta': 0.3827336180594706, 'c': 0.5803929422742778, 'mu': 0.4493971165453735}. Best is trial 3 with value: 0.4699330660549007.
[I 2026-06

In [5]:
# =============================================================================
# 5. POST-PROCESSING AND VISUALIZATION
# =============================================================================

df = study.trials_dataframe()
X_obs = torch.tensor(df[['params_rho', 'params_eta', 'params_c', 'params_mu']].values, dtype=torch.float64)
Y_obs = torch.tensor(df['value'].values, dtype=torch.float64).unsqueeze(-1)

# Train final GP model with standardizing transforms
model = SingleTaskGP(
    X_obs,
    Y_obs,
    input_transform=Normalize(d=X_obs.shape[-1]),
    outcome_transform=Standardize(m=1)
)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

# 2D Slice generation
ETA_FIXED = 0.5
MU_FIXED = 0.5
RESOLUTION = 100

rho_grid = np.linspace(0, 1, RESOLUTION)
c_grid = np.linspace(0, 1, RESOLUTION)
R, C = np.meshgrid(rho_grid, c_grid)

# Prepare predictive tensor
X_test = np.column_stack([
    R.ravel(),
    np.full_like(R.ravel(), ETA_FIXED),
    C.ravel(),
    np.full_like(R.ravel(), MU_FIXED)
])
X_test_tensor = torch.tensor(X_test, dtype=torch.float64)

# Retrieve predictions
model.eval()
with torch.no_grad():
    posterior = model.posterior(X_test_tensor)
    mu_pred = posterior.mean.numpy().squeeze()
    sigma_pred = posterior.variance.clamp_min(1e-9).sqrt().numpy().squeeze()

print("Process Complete.")

Process Complete.


### Exporting `X_obs` and `Y_obs`

YouYou can save these PyTorch tensors to `.pt` files using `torch.save()` for later use.

In [6]:
# Define filenames for saving
x_obs_filename = './X_obs.pt'
y_obs_filename = './Y_obs.pt'

# Save the tensors
torch.save(X_obs, x_obs_filename)
torch.save(Y_obs, y_obs_filename)

print(f"X_obs saved to {x_obs_filename}")
print(f"Y_obs saved to {y_obs_filename}")

X_obs saved to ./X_obs.pt
Y_obs saved to ./Y_obs.pt


In [8]:
# Define filenames for saving additional results
mu_pred_filename = './mu_pred.npy'
sigma_pred_filename = './sigma_pred.npy'

# Save the numpy arrays
np.save(mu_pred_filename, mu_pred)
np.save(sigma_pred_filename, sigma_pred)

print(f"mu_pred saved to {mu_pred_filename}")
print(f"sigma_pred saved to {sigma_pred_filename}")

mu_pred saved to ./mu_pred.npy
sigma_pred saved to ./sigma_pred.npy


In [9]:
df.to_csv('./optuna_study_results.csv', index=False)

In [14]:
import optuna
from optuna.storages import RDBStorage
import os

# Define the storage path for the exported study to Google Drive
storage_path = "sqlite:///optuna_study_exported.db"

# Extract the local file path from the sqlite URI
storage_path_local = storage_path.replace("sqlite:///", "")

# Ensure the directory exists
os.makedirs(os.path.dirname(storage_path_local), exist_ok=True)

# Create an RDBStorage object

exported_study = optuna.create_study(study_name="exported_graph_manifold_study", direction="minimize", storage=storage_path, load_if_exists=True)

# Copy trials from the in-memory study to the new persistent study
for trial in study.get_trials():
    try:
        exported_study.add_trial(trial)
    except Exception as e:
        print(f"Could not add trial {trial.number}: {e}")

print(f"Study exported to {storage_path}")

[I 2026-06-10 23:48:17,173] A new study created in RDB with name: exported_graph_manifold_study


Study exported to sqlite:///optuna_study_exported.db


In [15]:
# 1. Create a 3D dense grid (Resolution 20x20x20 = 8000 points)
RES = 20
grid_1d = np.linspace(0, 1, RES)
R, E, C = np.meshgrid(grid_1d, grid_1d, grid_1d)

# Set the 4th parameter (mu) to a fixed slice
X_test_3d = np.column_stack([
    R.ravel(),
    E.ravel(),
    C.ravel(),
    np.full_like(R.ravel(), 0.5) # Fixing Community Mixing
])
X_test_tensor_3d = torch.tensor(X_test_3d, dtype=torch.float64)

# 2. Get GP Predictions
model.eval()
with torch.no_grad():
    posterior = model.posterior(X_test_tensor_3d)
    mu_pred = posterior.mean.numpy().squeeze()
    sigma_pred = posterior.variance.clamp_min(1e-9).sqrt().numpy().squeeze()

# 3. Calculate Feasibility Probability
prob_feasible_3d = norm.cdf((0.01 - mu_pred) / sigma_pred)

# 4. Save to MATLAB format
sio.savemat('./gp_manifold_3d.mat', {
    'rho': R,
    'eta': E,
    'c': C,
    'probability': prob_feasible_3d.reshape(RES, RES, RES)
})
print("Saved gp_manifold_3d.mat for MATLAB!")

Saved gp_manifold_3d.mat for MATLAB!


In [16]:
# 1. Define the objective (Maximize Clustering)
def objective(x):
    return -x[2]  # Minimize -c to Maximize c

# 2. Define the constraint function
def constraint_fun(x):
    # Differential Evolution might pass a 1D array, we ensure it's formatted for BoTorch
    x_tensor = torch.tensor(x, dtype=torch.float64).unsqueeze(0)
    with torch.no_grad():
        pred_loss = model.posterior(x_tensor).mean.item()
    return pred_loss

# 3. Create a NonlinearConstraint (Loss must be between -infinity and 0.01)
nlc = NonlinearConstraint(constraint_fun, -np.inf, 0.01)

# 4. Set the bounds for [rho, eta, c, mu]
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

print("Deploying Differential Evolution to search the manifold...")

# 5. Run the gradient-free optimization
result = differential_evolution(
    objective,
    bounds,
    constraints=(nlc,),
    seed=42,        # For reproducibility
    popsize=15,     # Number of parallel agents searching
    maxiter=100     # Maximum generations
)

if result.success:
    ext = result.x
    print(f"\n✅ Boundary Found Successfully!")
    print(f"Absolute Limit of Clustering: c = {ext[2]:.4f}")
    print(f"Required coordinates at this extreme: rho={ext[0]:.4f}, eta={ext[1]:.4f}, mu={ext[3]:.4f}")
    print(f"Predicted Loss at this point: {constraint_fun(ext):.5f}")
else:
    print(f"Optimizer failed: {result.message}")

Deploying Differential Evolution to search the manifold...

✅ Boundary Found Successfully!
Absolute Limit of Clustering: c = 1.0000
Required coordinates at this extreme: rho=0.9841, eta=0.0034, mu=0.4912
Predicted Loss at this point: 0.00820


In [17]:
# 1. SAVE THE GAUSSIAN PROCESS MODEL
# We save the 'state_dict' (the learned mathematical weights)
# and the training data used to condition the GP.
torch.save({
    'model_state_dict': model.state_dict(),
    'X_train': X_obs,
    'Y_train': Y_obs
}, './botorch_gp_model.pth')
print("💾 GP Model saved to 'botorch_gp_model.pth'")

# 2. SAVE THE OPTIMIZATION RESULT
# We extract the important parts of the SciPy result (the coordinates and the loss)
extreme_data = {
    'extreme_coordinates': result.x,        # The 4D array [rho, eta, c, mu]
    'predicted_loss': constraint_fun(result.x),
    'message': result.message,
    'success': result.success
}

# Save as a standard Numpy .npz file (easy to load in Python or MATLAB)
np.savez('./extreme_boundary_result.npz', **extreme_data)
print("💾 Optimizer result saved to 'extreme_boundary_result.npz'")

💾 GP Model saved to 'botorch_gp_model.pth'
💾 Optimizer result saved to 'extreme_boundary_result.npz'
